In [ ]:
# Load necessary libraries
import numpy as np
import pandas as pd
import warnings
from scipy.spatial.transform import Rotation as R
from Bio.PDB import PDBParser
from Bio import PDB

warnings.filterwarnings("ignore")

In [3]:
# --- PDB Extraction Function ---
def extract_ca_coordinates(pdb_file):
    """
    Loads a PDB file and extracts C-alpha coordinates for each standard amino acid,
    returning them as a list of NumPy arrays, where each array is (1, 3) for a single residue.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    
    coords_per_residue = [] 
    
    for model in structure:
        for chain in model:
            for residue in chain:
                if PDB.is_aa(residue, standard=True) and residue.has_id('CA'):
                    coords_per_residue.append(np.array([residue['CA'].coord]))
    
    return coords_per_residue

In [4]:
# --- calculate_rmsd_after_superposition function ---
def calculate_rmsd_after_superposition(coords1, coords2):
    """
    Calculates the Root Mean Square Deviation (RMSD) between two sets of 3D coordinates
    after optimally superposing coords2 onto coords1 using the Kabsch algorithm (via scipy).

    Args:
        coords1 (np.ndarray): N x 3 array of 3D coordinates (reference).
        coords2 (np.ndarray): M x 3 array of 3D coordinates (to be superposed).

    Returns:
        float: The minimal RMSD between the two sets of coordinates.
    """
    # Robustness checks for empty arrays
    if coords1.shape[0] == 0 and coords2.shape[0] == 0:
        return 0.0 # Both empty, perfectly "aligned" (no cost)
    elif coords1.shape[0] == 0 or coords2.shape[0] == 0:
        # One is empty, the other is not. This should incur a high cost.
        return 1000.0 # A high penalty, indicating a fundamental mismatch
    
    # Handle single-point comparison directly (avoids R.align_vectors issues for N=1)
    if coords1.shape[0] == 1 and coords2.shape[0] == 1:
        return np.linalg.norm(coords1[0] - coords2[0]) # Euclidean distance for 1 point is its RMSD

    # Standard sanity check for 3D coordinates
    if coords1.shape[1] != 3 or coords2.shape[1] != 3:
        raise ValueError("Input coordinate arrays must be (N, 3) or (M, 3).")

    # Center the coordinates
    centroid1 = np.mean(coords1, axis=0)
    centroid2 = np.mean(coords2, axis=0)
    centered_coords1 = coords1 - centroid1
    centered_coords2 = coords2 - centroid2

    # --- Check for degenerate (all points identical / zero length after centering) inputs ---
    # If the sum of squared magnitudes of the centered vectors is effectively zero,
    # it means all points are identical, and alignment is trivial (RMSD is 0).
    is_coords1_degenerate = np.isclose(np.sum(centered_coords1**2), 0.0)
    is_coords2_degenerate = np.isclose(np.sum(centered_coords2**2), 0.0)

    if is_coords1_degenerate and is_coords2_degenerate:
        return 0.0 # Both sets of points are effectively identical (or collapsed to a single point)
    elif is_coords1_degenerate or is_coords2_degenerate:
        # One set is degenerate (e.g., all its atoms are at the same coordinate),
        # while the other is not. This is a significant structural mismatch.
        return 1000.0 # High penalty

    # Find the optimal rotation (will only be called if N > 1 AND not degenerate)
    rotation, rmsd = R.align_vectors(centered_coords2, centered_coords1)

    return rmsd


In [5]:
# --- dtw_with_rmsd_cost function ---
def dtw_with_rmsd_cost(seq1_coords, seq2_coords):
    """
    Performs Dynamic Time Warping (DTW) on two sequences of 3D backbone coordinates,
    using RMSD as the local cost metric between corresponding residues.

    Args:
        seq1_coords (list of np.ndarray): List where each element is a (1, 3) or (N_atoms, 3)
                                          numpy array representing the coordinates for one residue/segment.
                                          For simple C-alpha, it's (1, 3).
        seq2_coords (list of np.ndarray): Similar list for the second sequence.

    Returns:
        tuple: (dtw_cost, warping_path)
            dtw_cost (float): The total accumulated cost of the optimal warping path.
            warping_path (list): A list of (index_seq1, index_seq2) tuples representing
                                 the optimal alignment path.
    """
    n = len(seq1_coords)
    m = len(seq2_coords)

    # Handle cases where one or both sequences are empty
    if n == 0 and m == 0:
        return 0.0, []
    elif n == 0 or m == 0:
        return np.inf, [] # Infinite cost if one sequence is empty and the other is not

    # Initialize cost matrix
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0

    # Fill the cost matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            local_cost = calculate_rmsd_after_superposition(
                seq1_coords[i-1].reshape(-1, 3), # Ensure it's (N_atoms, 3)
                seq2_coords[j-1].reshape(-1, 3)  # Ensure it's (N_atoms, 3)
            )

            # Accumulate cost from previous cells
            D[i, j] = local_cost + min(D[i-1, j],    # Deletion (move right in seq1)
                                      D[i, j-1],    # Insertion (move down in seq2)
                                      D[i-1, j-1])  # Match/Substitution (diagonal)

    # Traceback to find the optimal path
    path = []
    i, j = n, m
    while i > 0 or j > 0:
        path.append((i - 1, j - 1)) # Append 0-indexed values
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            min_prev = min(D[i-1, j-1], D[i-1, j], D[i, j-1])
            if min_prev == D[i-1, j-1]:
                i -= 1
                j -= 1
            elif min_prev == D[i-1, j]:
                i -= 1
            else: # min_prev == D[i, j-1]:
                j -= 1
    path.reverse() # Path is built backwards, so reverse it

    total_cost = D[n, m]
    length_of_warping_path = len(path)

    if length_of_warping_path > 0:
        normalized_dtw_cost = total_cost / length_of_warping_path
    else:
        # This case should ideally not happen if n > 0 or m > 0 due to path construction,
        # but as a safeguard
        normalized_dtw_cost = np.inf if (n > 0 or m > 0) else 0.0

    return normalized_dtw_cost, path

In [ ]:
# --- Example Usage ---

# Load PDB and extract coordinates
seq1_coords = extract_ca_coordinates('data/1avw.pdb')
seq2_coords = extract_ca_coordinates('data/1cfd.pdb')

print(f"Sequence 1 has {len(seq1_coords)} residues.")
print(f"Sequence 2 has {len(seq2_coords)} residues.")

if seq1_coords and seq2_coords: # Proceed only if both sequences have data
    total_cost, path = dtw_with_rmsd_cost(seq1_coords, seq2_coords)

    print(f"\nTotal DTW Cost: {total_cost:.4f} Angstroms")
    # --- Code to calculate Normalized DTW Distance ---
    length_of_warping_path = len(path)
    if length_of_warping_path > 0:
        normalized_dtw_distance = total_cost / length_of_warping_path
        print(f"Normalized DTW Distance (by Warping Length Path): {normalized_dtw_distance:.4f} Angstroms")
    else:
        print("Warning: Warping path has zero length, cannot normalize DTW distance.")

    print("\nOptimal Warping Path:")
    for i, (idx1, idx2) in enumerate(path):
        print(f"  Path Step {i+1}: seq1_residue[{idx1}] <-> seq2_residue[{idx2}]")
    if len(path) > 100:
        print("  ...")
else:
    print("\nError: One or both PDB files resulted in empty C-alpha coordinate lists.")
    print("Please check the PDB files and the `extract_ca_coordinates` function's filtering conditions.")


Sequence 1 has 394 residues.
Sequence 2 has 148 residues.

Total DTW Cost: 12004.2280 Angstroms
Normalized DTW Distance (by Warping Length Path): 30.4676 Angstroms

Optimal Warping Path:
  Path Step 1: seq1_residue[0] <-> seq2_residue[0]
  Path Step 2: seq1_residue[1] <-> seq2_residue[1]
  Path Step 3: seq1_residue[2] <-> seq2_residue[2]
  Path Step 4: seq1_residue[3] <-> seq2_residue[3]
  Path Step 5: seq1_residue[4] <-> seq2_residue[4]
  Path Step 6: seq1_residue[5] <-> seq2_residue[5]
  Path Step 7: seq1_residue[6] <-> seq2_residue[6]
  Path Step 8: seq1_residue[7] <-> seq2_residue[7]
  Path Step 9: seq1_residue[8] <-> seq2_residue[8]
  Path Step 10: seq1_residue[9] <-> seq2_residue[9]
  Path Step 11: seq1_residue[10] <-> seq2_residue[10]
  Path Step 12: seq1_residue[11] <-> seq2_residue[11]
  Path Step 13: seq1_residue[12] <-> seq2_residue[12]
  Path Step 14: seq1_residue[13] <-> seq2_residue[13]
  Path Step 15: seq1_residue[14] <-> seq2_residue[14]
  Path Step 16: seq1_residue[15]